# DishDash - Milestone 4: Filtering, Multi-Neighborhood Coverage, and Visualization

Builds on Milestone 3-5. New in this notebook:

1. **Multi-neighborhood data collection** - fetch OSM + Google Places across several NYC neighborhoods (NYU/Greenwich Village, East Village, SoHo, Lower East Side, Chinatown, Williamsburg, Upper West Side).
2. **Search & filter logic** - query the merged dataset by cuisine, price, rating, nutrition, sentiment, and neighborhood.
3. **Folium map of Manhattan** for the geographic distribution, plus seaborn charts for the rest of the analytics.


---
## 0. Setup & Installs

First, we install the libraries we need and set up our API keys.  
**Everyone in the group can run this cell first.**

In [ ]:
# Install libraries not pre-installed in Colab
!pip install thefuzz python-Levenshtein --quiet

print("All packages installed!")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import requests
import pandas as pd
import sqlite3
import json
import time
import math
from thefuzz import fuzz

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 40)

print("All imports loaded!")

In [ ]:
# ============================================================
# API KEYS
# ============================================================
GOOGLE_API_KEY    = ""
HUGGINGFACE_TOKEN = ""

# ============================================================
# NEIGHBORHOODS - bounding boxes (S, W, N, E) + center for Google search
# ============================================================
NEIGHBORHOODS = {
    "NYU / Greenwich Village": {
        "bbox":   (40.7240, -74.0020, 40.7380, -73.9900),
        "center": (40.7295, -73.9965), "radius": 800},
    "East Village": {
        "bbox":   (40.7220, -73.9900, 40.7340, -73.9750),
        "center": (40.7280, -73.9830), "radius": 800},
    "SoHo": {
        "bbox":   (40.7200, -74.0080, 40.7280, -73.9950),
        "center": (40.7240, -74.0020), "radius": 700},
    "Lower East Side": {
        "bbox":   (40.7140, -73.9950, 40.7240, -73.9800),
        "center": (40.7190, -73.9870), "radius": 800},
    "Chinatown": {
        "bbox":   (40.7100, -74.0050, 40.7180, -73.9900),
        "center": (40.7158, -73.9970), "radius": 600},
    "Williamsburg": {
        "bbox":   (40.7050, -73.9650, 40.7200, -73.9450),
        "center": (40.7130, -73.9580), "radius": 900},
    "Upper West Side": {
        "bbox":   (40.7780, -73.9900, 40.7920, -73.9700),
        "center": (40.7850, -73.9760), "radius": 900},
}
print(f"Configured {len(NEIGHBORHOODS)} neighborhoods:")
for name in NEIGHBORHOODS:
    print(f"  - {name}")


In [ ]:
# Quick smoke-test of the Google Places API key (skipped if blank)
if GOOGLE_API_KEY:
    resp = requests.post(
        "https://places.googleapis.com/v1/places:searchNearby",
        headers={
            "Content-Type": "application/json",
            "X-Goog-Api-Key": GOOGLE_API_KEY,
            "X-Goog-FieldMask": "places.displayName"},
        json={"locationRestriction": {"circle":
                {"center": {"latitude": 40.7295, "longitude": -73.9965},
                 "radius": 500}},
              "includedTypes": ["restaurant"], "maxResultCount": 1})
    print(f"Status: {resp.status_code}")
    print(resp.text[:200])
else:
    print("GOOGLE_API_KEY is blank - skipping smoke test")


---
## 1. Data Source #1 — Overpass API (OpenStreetMap)

**What is it?** OpenStreetMap is like Wikipedia for maps — community-edited geographic data.  
The Overpass API lets us query OSM data with filters.

**What we get:** Every restaurant, cafe, and fast food place near NYU with
name, lat/lng, cuisine type, address, phone, website, and hours.

**Auth required?** None!

In [ ]:
def fetch_osm_restaurants(bbox_s, bbox_w, bbox_n, bbox_e, max_retries=3):
    """Query the Overpass API with automatic retries on timeout.

    Uses POST + a real User-Agent header. Overpass returns HTTP 406 for
    requests with the default `python-requests/...` UA, so identifying
    the client is required. Falls back to a mirror if the main endpoint
    keeps failing.
    """

    overpass_query = f"""
    [out:json][timeout:60];
    (
      node["amenity"="restaurant"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
      node["amenity"="cafe"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
      node["amenity"="fast_food"]({bbox_s},{bbox_w},{bbox_n},{bbox_e});
    );
    out body;
    """

    ENDPOINTS = [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",   # mirror fallback
        "https://overpass.private.coffee/api/interpreter", # second mirror
    ]
    HEADERS = {
        "User-Agent": "DishDash-Student-Project/1.0 (NYU Stern TECH-UB.0024; contact: yc6393@nyu.edu)",
        "Accept": "application/json",
    }

    for attempt in range(1, max_retries + 1):
        url = ENDPOINTS[(attempt - 1) % len(ENDPOINTS)]
        print(f"Querying Overpass API (attempt {attempt}/{max_retries}) -> {url}")
        try:
            # POST with the query in the body is the recommended form for
            # large queries and is friendlier to Overpass rate limiters.
            response = requests.post(
                url,
                data={"data": overpass_query},
                headers=HEADERS,
                timeout=90,
            )

            if response.status_code == 200:
                elements = response.json().get("elements", [])
                print(f"Found {len(elements)} places from OSM")

                records = []
                for el in elements:
                    tags = el.get("tags", {})
                    records.append({
                        "osm_id":        el.get("id"),
                        "name":          tags.get("name", "Unknown"),
                        "lat":           el.get("lat"),
                        "lng":           el.get("lon"),
                        "amenity_type":  tags.get("amenity", ""),
                        "cuisine":       tags.get("cuisine", "unknown"),
                        "addr_street":   tags.get("addr:street", ""),
                        "addr_number":   tags.get("addr:housenumber", ""),
                        "phone":         tags.get("phone", ""),
                        "website":       tags.get("website", ""),
                        "opening_hours": tags.get("opening_hours", ""),
                        "wheelchair":    tags.get("wheelchair", ""),
                    })

                df = pd.DataFrame(records)
                df = df[df["name"] != "Unknown"].reset_index(drop=True)
                print(f"{len(df)} named places after cleaning")
                return df

            print(f"Error: status code {response.status_code} — {response.text[:200]}")

        except requests.exceptions.Timeout:
            print(f"Timeout on attempt {attempt}")
        except requests.exceptions.RequestException as e:
            print(f"Request failed on attempt {attempt}: {e}")

        if attempt < max_retries:
            wait = attempt * 10
            print(f"Waiting {wait}s before retry...")
            time.sleep(wait)

    print("All retries failed. Returning empty DataFrame.")
    return pd.DataFrame()


In [ ]:
# Fetch OSM data for every neighborhood and tag each row.
osm_frames = []
for name, cfg in NEIGHBORHOODS.items():
    print(f"\n--- {name} ---")
    s, w, n, e = cfg["bbox"]
    df_n = fetch_osm_restaurants(s, w, n, e)
    if len(df_n) > 0:
        df_n["neighborhood"] = name
        osm_frames.append(df_n)
df_osm = pd.concat(osm_frames, ignore_index=True) if osm_frames else pd.DataFrame()
if len(df_osm) > 0 and "osm_id" in df_osm.columns:
    df_osm = df_osm.drop_duplicates(subset=["osm_id"]).reset_index(drop=True)
print(f"\n{'='*60}")
print("OSM DATA SUMMARY (all neighborhoods)")
print(f"{'='*60}")
print(f"Total restaurants: {len(df_osm)}")
if len(df_osm) > 0:
    print("\nBy neighborhood:")
    print(df_osm["neighborhood"].value_counts())
    print("\nTop cuisines:")
    print(df_osm["cuisine"].value_counts().head(10))
    display(df_osm.head())


---
## 2. Data Source #2 — Google Places API (New)

**What is it?** Google's Places API gives us access to Google Maps venue data —
ratings, reviews, price levels, business status, and more.

**What we get:**
- Business name, location, formatted address
- Google rating (1.0-5.0) and total number of ratings
- Price level (PRICE_LEVEL_INEXPENSIVE to PRICE_LEVEL_VERY_EXPENSIVE)
- Types/categories
- User reviews (text + individual rating)

**Auth required?** Yes — Google Cloud API key with Places API (New) enabled.

**How the new API works:**  
The Places API (New) uses **POST requests** with JSON bodies, and you control
which fields are returned (and billed) using a **FieldMask** header. We request
only the fields we need to stay within the free tier.

**Setup (5 minutes):**
1. Go to https://console.cloud.google.com
2. Create a project -> Enable "Places API (New)"
3. Create an API key under Credentials
4. Paste it in Section 0 above

In [ ]:
def fetch_google_places(lat, lng, radius_m, api_key, max_results=20):
    """
    Search for restaurants near a location using Google Places API (New).

    Uses the Nearby Search endpoint (POST request).
    The FieldMask header controls which fields we get back —
    and what we get billed for. We stick to basic + a few pro fields.

    Parameters:
        lat, lng:      center point coordinates
        radius_m:      search radius in meters
        api_key:       your Google Cloud API key
        max_results:   max results (Google caps at 20 per request)

    Returns:
        pandas DataFrame with venue data
    """

    if api_key == "YOUR_GOOGLE_KEY_HERE":
        print("Please set your Google API key in Section 0!")
        return pd.DataFrame()

    url = "https://places.googleapis.com/v1/places:searchNearby"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        # FieldMask: controls which fields are returned
        # displayName, location, formattedAddress = Essentials (free 10K/mo)
        # rating, userRatingCount, priceLevel, types = Pro (free 5K/mo)
        # reviews = Pro
        "X-Goog-FieldMask": ",".join([
            "places.id",
            "places.displayName",
            "places.location",
            "places.formattedAddress",
            "places.types",
            "places.rating",
            "places.userRatingCount",
            "places.priceLevel",
            "places.reviews",
            "places.businessStatus",
        ])
    }

    # Request body for Nearby Search
    body = {
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": lat,
                    "longitude": lng
                },
                "radius": radius_m
            }
        },
        "includedTypes": ["restaurant", "cafe", "fast_food_restaurant",
                          "pizza_restaurant", "coffee_shop", "meal_takeaway"],
        "maxResultCount": max_results,
        "languageCode": "en",
    }

    print("Querying Google Places API (New)...")
    response = requests.post(url, headers=headers, json=body)

    if response.status_code != 200:
        print(f"Error: status code {response.status_code}")
        error_data = response.json() if response.text else {}
        error_msg = error_data.get("error", {}).get("message", response.text[:300])
        print(f"Message: {error_msg}")
        return pd.DataFrame()

    data = response.json()
    places = data.get("places", [])
    print(f"Found {len(places)} places from Google")

    records = []
    for place in places:
        # Extract display name
        display_name = place.get("displayName", {}).get("text", "Unknown")

        # Extract location
        loc = place.get("location", {})

        # Extract first review text (if any)
        reviews = place.get("reviews", [])
        review_texts = []
        for r in reviews[:3]:  # up to 3 reviews
            txt = r.get("text", {}).get("text", "")
            if txt:
                review_texts.append(txt)
        combined_reviews = " | ".join(review_texts)

        # Map Google's price level enum to a readable format
        price_map = {
            "PRICE_LEVEL_FREE": "Free",
            "PRICE_LEVEL_INEXPENSIVE": "$",
            "PRICE_LEVEL_MODERATE": "$$",
            "PRICE_LEVEL_EXPENSIVE": "$$$",
            "PRICE_LEVEL_VERY_EXPENSIVE": "$$$$",
        }
        raw_price = place.get("priceLevel", "")
        price_level = price_map.get(raw_price, raw_price)

        # Get types (categories)
        types = place.get("types", [])
        # Filter out generic types to find the most useful one
        useful_types = [t for t in types if t not in
                        ("point_of_interest", "establishment", "food")]
        primary_type = useful_types[0] if useful_types else (types[0] if types else "unknown")

        records.append({
            "google_place_id":  place.get("id", ""),
            "name":             display_name,
            "lat":              loc.get("latitude"),
            "lng":              loc.get("longitude"),
            "address":          place.get("formattedAddress", ""),
            "google_type":      primary_type,
            "all_types":        ", ".join(types[:5]),
            "price_level":      price_level,
            "google_rating":    place.get("rating"),
            "rating_count":     place.get("userRatingCount", 0),
            "business_status":  place.get("businessStatus", ""),
            "review_text":      combined_reviews,
        })

    df = pd.DataFrame(records)
    df = df[df["name"] != "Unknown"].reset_index(drop=True)
    print(f"{len(df)} named venues after cleaning")
    return df

In [ ]:
# Fetch Google Places for every neighborhood and tag each row.
google_frames = []
for name, cfg in NEIGHBORHOODS.items():
    print(f"\n--- {name} ---")
    lat, lng = cfg["center"]
    df_n = fetch_google_places(lat, lng, cfg["radius"], GOOGLE_API_KEY)
    if len(df_n) > 0:
        df_n["neighborhood"] = name
        google_frames.append(df_n)
df_google = pd.concat(google_frames, ignore_index=True) if google_frames else pd.DataFrame()
if len(df_google) > 0 and "google_place_id" in df_google.columns:
    df_google = df_google.drop_duplicates(subset=["google_place_id"]).reset_index(drop=True)
if len(df_google) > 0:
    print(f"\n{'='*60}")
    print("GOOGLE PLACES DATA SUMMARY (all neighborhoods)")
    print(f"{'='*60}")
    print(f"Total venues: {len(df_google)}")
    print("\nBy neighborhood:")
    print(df_google["neighborhood"].value_counts())
    print(f"\nAvg rating: {df_google['google_rating'].mean():.1f}")
    print(f"Venues with reviews: {df_google['review_text'].astype(bool).sum()}")
    display(df_google.head())
else:
    print("\nNo Google data - did you set your API key?")


In [ ]:
# FALLBACK: If OSM returned nothing, use Google as the base instead
if len(df_osm) == 0 and len(df_google) > 0:
    print("OSM returned no data — using Google Places as the base instead.\n")
    df_osm = df_google.rename(columns={
        "google_place_id": "osm_id",
        "google_type": "amenity_type",
        "address": "addr_street",
    }).copy()
    df_osm['cuisine'] = df_osm.get('all_types', 'unknown')
    df_osm['addr_number'] = ''
    df_osm['phone'] = ''
    df_osm['website'] = ''
    df_osm['opening_hours'] = ''
    df_osm['wheelchair'] = ''
    print(f"Using {len(df_osm)} Google places as base DataFrame")
else:
    print(f"OSM has {len(df_osm)} places — proceeding normally")


---
## 3. Data Source #3 — Open Food Facts API

**What is it?** Open Food Facts is a free, open database of food products.
Think of it as Wikipedia for nutrition labels.

**What we get:** Nutritional info (calories, fat, protein, carbs, sugar, sodium),
Nutri-Score (A-E health grade), and allergen warnings.

**Our approach:** We can't look up a *restaurant* on Open Food Facts — it tracks
*products*, not venues. So we map each restaurant's **cuisine type** to a
representative food search, then compute average nutrition stats per cuisine.

**Auth required?** None!

In [ ]:
# ============================================================
# CUISINE -> FOOD SEARCH MAPPING
# ============================================================

CUISINE_TO_SEARCH = {
    "pizza":       ["pizza"],
    "italian":     ["pasta", "pizza", "risotto"],
    "chinese":     ["noodles", "fried rice", "dumplings"],
    "japanese":    ["sushi", "ramen", "miso"],
    "sushi":       ["sushi", "sashimi"],
    "mexican":     ["burrito", "taco", "nachos"],
    "burger":      ["burger", "hamburger"],
    "american":    ["burger", "sandwich", "fries"],
    "indian":      ["curry", "naan", "biryani"],
    "thai":        ["pad thai", "curry", "spring roll"],
    "mediterranean":["hummus", "falafel", "pita"],
    "french":      ["croissant", "baguette", "quiche"],
    "korean":      ["bibimbap", "kimchi", "bulgogi"],
    "vietnamese":  ["pho", "banh mi", "spring roll"],
    "sandwich":    ["sandwich", "sub"],
    "coffee_shop": ["coffee", "latte", "muffin"],
    "cafe":        ["coffee", "pastry", "sandwich"],
    "unknown":     ["meal", "food"],
}

print(f"Defined search terms for {len(CUISINE_TO_SEARCH)} cuisine types")

In [ ]:
def fetch_nutrition_for_cuisine(cuisine_key, search_terms, max_products=5):
    """Aggregate nutrition for a cuisine by querying Open Food Facts.

    Open Food Facts requires every client to send a descriptive User-Agent;
    requests without one (or with the default `python-requests/...`) are
    rejected. We also try the new v2 search endpoint if the legacy one
    returns a non-200, and we surface the failure instead of swallowing it.
    """
    HEADERS = {
        "User-Agent": "DishDash-Student-Project/1.0 (NYU Stern TECH-UB.0024; contact: yc6393@nyu.edu)",
        "Accept": "application/json",
    }
    ENDPOINTS = [
        "https://world.openfoodfacts.org/cgi/search.pl",
        "https://world.openfoodfacts.org/api/v2/search",   # newer v2 API
    ]

    all_nutrition = []

    for term in search_terms:
        # Try each endpoint up to 2 times; first success wins.
        got_any = False
        for url in ENDPOINTS:
            for attempt in range(2):
                params = {
                    "search_terms": term,
                    "json": 1,
                    "page_size": max_products,
                    "fields": "product_name,nutriscore_grade,nutriments,allergens_tags",
                }
                try:
                    resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
                    if resp.status_code != 200:
                        print(f"    {url} -> {resp.status_code} for '{term}'")
                        break  # try next endpoint
                    products = resp.json().get("products", [])
                    for p in products:
                        nuts = p.get("nutriments", {}) or {}
                        # Some products report energy in kJ only; convert.
                        kcal = nuts.get("energy-kcal_100g")
                        if kcal is None:
                            kj = nuts.get("energy-kj_100g") or nuts.get("energy_100g")
                            if kj is not None:
                                kcal = kj / 4.184
                        if kcal is None:
                            continue
                        all_nutrition.append({
                            "calories_100g": kcal,
                            "fat_100g":      nuts.get("fat_100g") or 0,
                            "protein_100g":  nuts.get("proteins_100g") or 0,
                            "carbs_100g":    nuts.get("carbohydrates_100g") or 0,
                            "sugar_100g":    nuts.get("sugars_100g") or 0,
                            "sodium_100g":   nuts.get("sodium_100g") or 0,
                            "nutriscore":    p.get("nutriscore_grade", ""),
                        })
                    got_any = True
                    break  # success on this endpoint
                except requests.exceptions.RequestException as e:
                    if attempt == 0:
                        print(f"    Retrying '{term}' in 5s...  ({e})")
                        time.sleep(5)
                    else:
                        print(f"    Skipped '{term}' (network error: {e})")
            if got_any:
                break  # don't try the next endpoint
        time.sleep(1)  # Polite pause between terms

    if not all_nutrition:
        return {
            "cuisine_key": cuisine_key,
            "avg_calories_100g": None, "avg_fat_100g": None,
            "avg_protein_100g": None, "avg_carbs_100g": None,
            "avg_sugar_100g": None, "avg_sodium_100g": None,
            "common_nutriscore": None, "off_sample_size": 0
        }

    df_temp = pd.DataFrame(all_nutrition)
    nutri_scores = df_temp["nutriscore"].replace("", pd.NA).dropna()
    common_ns = nutri_scores.mode().iloc[0] if len(nutri_scores) > 0 else None

    return {
        "cuisine_key":       cuisine_key,
        "avg_calories_100g": round(df_temp["calories_100g"].mean(), 1),
        "avg_fat_100g":      round(df_temp["fat_100g"].mean(), 1),
        "avg_protein_100g":  round(df_temp["protein_100g"].mean(), 1),
        "avg_carbs_100g":    round(df_temp["carbs_100g"].mean(), 1),
        "avg_sugar_100g":    round(df_temp["sugar_100g"].mean(), 1),
        "avg_sodium_100g":   round(df_temp["sodium_100g"].mean(), 3),
        "common_nutriscore": common_ns,
        "off_sample_size":   len(all_nutrition),
    }


In [ ]:
# Fetch nutrition data for ALL cuisine types
print("Fetching nutritional data from Open Food Facts...")
print("(This takes ~2 min — we add delays to be polite to the API)\n")

nutrition_records = []
for cuisine, terms in CUISINE_TO_SEARCH.items():
    print(f"  Searching: {cuisine} -> {terms}")
    result = fetch_nutrition_for_cuisine(cuisine, terms)
    nutrition_records.append(result)
    time.sleep(1)

df_nutrition = pd.DataFrame(nutrition_records)

print(f"\n{'='*60}")
print("OPEN FOOD FACTS SUMMARY")
print(f"{'='*60}")
print(f"Cuisine profiles collected: {len(df_nutrition)}")
print(f"Cuisines with data: {df_nutrition['avg_calories_100g'].notna().sum()}")
print()
df_nutrition


---
## 4. Merging the Data Sources

Now the key part — combining our 3 data sources into one unified DataFrame.

### Merge Strategy

**Step A: OSM + Google Places (fuzzy geo-match)**  
Both sources have restaurant names and coordinates. We match them using:
1. **Geographic proximity** — within ~75 meters of each other?
2. **Name similarity** — fuzzy string matching on names

LEFT JOIN approach: keep ALL OSM restaurants, even without a Google match.

**Step B: Merged venues + Nutrition (cuisine category)**  
LEFT JOIN on the normalized `cuisine_key` — every "pizza" restaurant gets
the same nutrition profile.

In [ ]:
def haversine_meters(lat1, lng1, lat2, lng2):
    """Distance in meters between two lat/lng points (Haversine formula)."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lng2 - lng1)
    a = (math.sin(dphi / 2) ** 2 +
         math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Quick test
d = haversine_meters(40.7295, -73.9965, 40.7300, -73.9960)
print(f"Test distance: {d:.1f} meters (should be ~70m)")

In [ ]:
def fuzzy_geo_merge(df_osm, df_google, max_distance_m=75, min_name_score=55):
    """
    Match OSM restaurants to Google Places using:
    1. Geographic proximity (within max_distance_m meters)
    2. Fuzzy name matching (score >= min_name_score out of 100)

    LEFT JOIN: all OSM rows are kept, Google data added where matched.
    """

    google_cols = ['google_place_id', 'google_type', 'all_types', 'price_level',
                   'google_rating', 'rating_count', 'review_text',
                   'match_score', 'match_distance_m']

    if len(df_google) == 0:
        print("No Google data — returning OSM data only.")
        for col in google_cols:
            df_osm[col] = None
        return df_osm

    print(f"Fuzzy matching {len(df_osm)} OSM places with {len(df_google)} Google venues...")

    matches = []
    match_count = 0

    for _, osm_row in df_osm.iterrows():
        best_match = None
        best_score = 0

        for _, g_row in df_google.iterrows():
            # Step 1: Geographic distance
            dist = haversine_meters(
                osm_row['lat'], osm_row['lng'],
                g_row['lat'], g_row['lng']
            )
            if dist > max_distance_m:
                continue

            # Step 2: Fuzzy name matching
            name_score = fuzz.token_sort_ratio(
                osm_row['name'].lower(),
                g_row['name'].lower()
            )
            if name_score < min_name_score:
                continue

            # Combined score: name similarity weighted more than distance
            combined_score = name_score * 0.7 + (1 - dist / max_distance_m) * 30

            if combined_score > best_score:
                best_score = combined_score
                best_match = {
                    "google_place_id": g_row['google_place_id'],
                    "google_type":     g_row['google_type'],
                    "all_types":       g_row['all_types'],
                    "price_level":     g_row['price_level'],
                    "google_rating":   g_row['google_rating'],
                    "rating_count":    g_row['rating_count'],
                    "review_text":     g_row.get('review_text', ''),
                    "match_score":     round(combined_score, 1),
                    "match_distance_m": round(dist, 1),
                }

        if best_match:
            matches.append(best_match)
            match_count += 1
        else:
            matches.append({col: None for col in google_cols})

    df_matches = pd.DataFrame(matches)
    df_merged = pd.concat([df_osm.reset_index(drop=True), df_matches], axis=1)

    print(f"Matched {match_count}/{len(df_osm)} OSM places to Google venues")
    print(f"({len(df_osm) - match_count} OSM-only, no Google match)")
    return df_merged

In [ ]:
# Run the OSM + Google merge
df_venues = fuzzy_geo_merge(df_osm, df_google)

print(f"\nMerged venue DataFrame: {df_venues.shape[0]} rows x {df_venues.shape[1]} columns")
df_venues.head()

In [ ]:
# ============================================================
# Normalize cuisine keys for the nutrition merge
# ============================================================

def normalize_cuisine(cuisine_raw):
    """Normalize OSM cuisine tag into a key matching CUISINE_TO_SEARCH."""
    if not cuisine_raw or cuisine_raw == "unknown":
        return "unknown"

    primary = cuisine_raw.split(";")[0].strip().lower()
    primary = primary.replace("-", "_").replace(" ", "_")

    if primary in CUISINE_TO_SEARCH:
        return primary

    aliases = {
        "coffee": "coffee_shop", "espresso": "coffee_shop",
        "sushi": "sushi", "ramen": "japanese",
        "tacos": "mexican", "burritos": "mexican",
        "kebab": "mediterranean", "falafel": "mediterranean",
        "pho": "vietnamese", "noodle": "chinese",
    }
    for alias, mapped in aliases.items():
        if alias in primary:
            return mapped
    return "unknown"

df_venues["cuisine_key"] = df_venues["cuisine"].apply(normalize_cuisine)

print("Normalized cuisine distribution:")
print(df_venues["cuisine_key"].value_counts())

In [ ]:
# ============================================================
# Merge venues with nutrition data (LEFT JOIN on cuisine_key)
# ============================================================

print(f"Venues shape BEFORE nutrition merge: {df_venues.shape}")

df_final = df_venues.merge(
    df_nutrition, on="cuisine_key", how="left", suffixes=('', '_nutrition')
)

print(f"Final shape AFTER nutrition merge: {df_final.shape}")
print(f"\nMERGE COMPLETE!")
print(f"{df_final.shape[0]} restaurants x {df_final.shape[1]} features")
print(f"Venues with nutrition data: {df_final['avg_calories_100g'].notna().sum()}")

df_final.head()

---
## 5. (Bonus) Sentiment Analysis on Reviews — HuggingFace

For venues that have Google review text, we run NLP sentiment analysis
using HuggingFace's free Inference API.

**Optional for Milestone 3** — gives you a head start on M4/M5.

In [ ]:
def analyze_sentiment_hf(text, hf_token):
    """Run sentiment analysis using HuggingFace Inference API."""
    if not text or text.strip() == "":
        return {"sentiment_label": None, "sentiment_score": None}

    API_URL = "https://api-inference.huggingface.co/models/distilbert-base-uncased-finetuned-sst-2-english"
    headers = {"Authorization": f"Bearer {hf_token}"}

    try:
        response = requests.post(API_URL, headers=headers,
                                 json={"inputs": text[:512]}, timeout=15)
        if response.status_code == 200:
            result = response.json()
            if result and isinstance(result, list) and len(result) > 0:
                top = result[0][0] if isinstance(result[0], list) else result[0]
                return {
                    "sentiment_label": top.get("label"),
                    "sentiment_score": round(top.get("score", 0), 3),
                }
    except Exception as e:
        print(f"  Sentiment API error: {e}")

    return {"sentiment_label": None, "sentiment_score": None}


if HUGGINGFACE_TOKEN != "YOUR_HF_TOKEN_HERE":
    review_count = df_final["review_text"].fillna("").str.strip().astype(bool).sum()
    print(f"Running sentiment analysis on {review_count} reviews...")

    sentiments = []
    for _, row in df_final.iterrows():
        text = row.get("review_text", "")
        if text and str(text).strip():
            result = analyze_sentiment_hf(str(text), HUGGINGFACE_TOKEN)
            time.sleep(0.3)
        else:
            result = {"sentiment_label": None, "sentiment_score": None}
        sentiments.append(result)

    df_sentiment = pd.DataFrame(sentiments)
    df_final["sentiment_label"] = df_sentiment["sentiment_label"]
    df_final["sentiment_score"] = df_sentiment["sentiment_score"]
    print(f"\nSentiment analysis complete!")
    print(df_final["sentiment_label"].value_counts())
else:
    print("Skipping sentiment analysis (no HuggingFace token set)")
    df_final["sentiment_label"] = None
    df_final["sentiment_score"] = None

---
## 6. Explore the Final Merged DataFrame

In [ ]:
print("=" * 70)
print("         DISHDASH — FINAL MERGED DATASET SUMMARY")
print("=" * 70)
print(f"\nShape: {df_final.shape[0]} restaurants x {df_final.shape[1]} columns\n")

print("ALL COLUMNS:")
print("-" * 50)
for i, col in enumerate(df_final.columns):
    source = "[OSM]" if col in ['osm_id','amenity_type','cuisine','addr_street',
                                 'addr_number','phone','website','opening_hours',
                                 'wheelchair'] else \
             "[GOOG]" if col in ['google_place_id','google_type','all_types',
                                  'price_level','google_rating','rating_count',
                                  'review_text','match_score','match_distance_m',
                                  'business_status'] else \
             "[OFF]" if col.startswith('avg_') or col in ['common_nutriscore',
                                                          'off_sample_size'] else \
             "[NLP]" if 'sentiment' in col else \
             "[shared]"
    non_null = df_final[col].notna().sum()
    print(f"  {i+1:2d}. {source:8s} {col:25s} ({non_null}/{len(df_final)} non-null)")

print(f"\nDATA SOURCE COVERAGE:")
print(f"   OSM restaurants:       {len(df_final)}")
print(f"   + Google Places match: {df_final['google_place_id'].notna().sum()}")
print(f"   + Nutrition data:      {df_final['avg_calories_100g'].notna().sum()}")
print(f"   + Sentiment scores:    {df_final['sentiment_label'].notna().sum()}")

In [ ]:
# Sample view of key columns
cols_to_show = ['name', 'cuisine_key', 'lat', 'lng', 'google_type',
                'google_rating', 'price_level', 'avg_calories_100g',
                'common_nutriscore', 'sentiment_label']
cols_to_show = [c for c in cols_to_show if c in df_final.columns]

print("SAMPLE DATA (key columns):")
df_final[cols_to_show].head(10)

---
## 7. Store in SQLite Database

SQLite is a lightweight database in a single file. We create 3 tables:
1. `venues` — main merged table
2. `nutrition_profiles` — cuisine-level nutrition lookup
3. `data_sources` — metadata documenting our data sources

In [ ]:
DB_FILENAME = "dishdash.db"
conn = sqlite3.connect(DB_FILENAME)
print(f"Created SQLite database: {DB_FILENAME}")

# Table 1: venues
df_final.to_sql(name='venues', con=conn, if_exists='replace', index=False)
print(f"Table 'venues': {len(df_final)} rows written")

# Table 2: nutrition_profiles
df_nutrition.to_sql(name='nutrition_profiles', con=conn, if_exists='replace', index=False)
print(f"Table 'nutrition_profiles': {len(df_nutrition)} rows written")

# Table 3: data_sources
data_sources = pd.DataFrame([
    {"source_name": "OpenStreetMap (Overpass API)",
     "url": "https://overpass-api.de/api/interpreter",
     "data_type": "Geospatial restaurant POI data",
     "auth_required": "No",
     "records_fetched": len(df_osm)},
    {"source_name": "Google Places API (New)",
     "url": "https://places.googleapis.com/v1/places:searchNearby",
     "data_type": "Ratings, reviews, price levels, categories",
     "auth_required": "Yes (API key, free tier)",
     "records_fetched": len(df_google)},
    {"source_name": "Open Food Facts",
     "url": "https://world.openfoodfacts.org/cgi/search.pl",
     "data_type": "Nutritional data by food category",
     "auth_required": "No",
     "records_fetched": len(df_nutrition)},
])

data_sources.to_sql(name='data_sources', con=conn, if_exists='replace', index=False)
print(f"Table 'data_sources': {len(data_sources)} rows written")

conn.commit()
print(f"\nAll data committed to {DB_FILENAME}")

In [ ]:
# ============================================================
# Verify the database
# ============================================================

print("VERIFYING DATABASE CONTENTS\n")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"Tables in database: {list(tables['name'])}\n")

for table_name in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table_name}", conn)
    print(f"  {table_name}: {count['cnt'].iloc[0]} rows")

print("\n" + "=" * 60)
print("SAMPLE QUERY: Top-rated restaurants with nutrition data")
print("=" * 60)

query = """
SELECT name, cuisine_key, google_rating, price_level,
       avg_calories_100g, common_nutriscore
FROM venues
WHERE google_rating IS NOT NULL
  AND avg_calories_100g IS NOT NULL
ORDER BY google_rating DESC
LIMIT 10
"""

pd.read_sql(query, conn)

In [ ]:
# Average nutrition by cuisine
print("SAMPLE QUERY: Average calories by cuisine")
print("=" * 60)

query2 = """
SELECT cuisine_key,
       COUNT(*) as num_restaurants,
       ROUND(AVG(avg_calories_100g), 1) as avg_cal,
       common_nutriscore
FROM venues
WHERE cuisine_key != 'unknown'
  AND avg_calories_100g IS NOT NULL
GROUP BY cuisine_key
ORDER BY avg_cal DESC
"""

pd.read_sql(query2, conn)

In [ ]:
conn.close()
print("Database connection closed.")
print(f"\nYour database file '{DB_FILENAME}' is saved in this Colab session.")
print("To download: click the folder icon (left sidebar) -> find dishdash.db -> download")

---
## 8. (Optional) Save to Google Drive

Colab sessions are temporary. Save to Google Drive to persist your database.

In [ ]:
# Uncomment to save to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('dishdash.db', '/content/drive/MyDrive/dishdash.db')
# print("Database saved to Google Drive!")

---
## Summary & Next Steps

### What We Did (Milestone 3)
- **3 data sources** pulled via APIs: OSM (Overpass), Google Places, Open Food Facts
- **Merged** into a single DataFrame using fuzzy geo-matching + cuisine-key joins
- **Stored** everything in a SQLite database with 3 tables
- **Bonus**: HuggingFace sentiment analysis on Google reviews

### Milestone 4 Ideas (Minimum Viable Unit)
- Build filter/search logic: by cuisine, price, nutrition, sentiment
- Add more neighborhoods beyond NYU area
- Create visualizations: map of restaurants, nutrition comparisons

### Milestone 5 Ideas (Minimal Frontend)
- Flask or Streamlit web app
- Interactive map (Folium or Mapbox)
- Search bar + filter sidebar
- Restaurant detail pages with nutrition cards

---

## 6. Search & Filter Logic

Query the merged venue dataset (`df_final`) by any combination of: cuisine key, price level, Google rating, calorie ceiling, Nutri-Score, sentiment, and neighborhood.


In [ ]:
PRICE_RANK = {"Free": 0, "$": 1, "$$": 2, "$$$": 3, "$$$$": 4}

def filter_venues(df, cuisine=None, neighborhood=None, max_price=None,
                  min_rating=None, max_calories=None, nutriscore=None,
                  sentiment=None, has_reviews=False,
                  sort_by="google_rating", ascending=False, limit=None):
    """Return a filtered + sorted view of df_final."""
    out = df.copy()
    def _as_list(v): return [v] if isinstance(v, str) else list(v)
    if cuisine is not None:
        out = out[out["cuisine_key"].isin(_as_list(cuisine))]
    if neighborhood is not None:
        out = out[out["neighborhood"].isin(_as_list(neighborhood))]
    if max_price is not None:
        cap = PRICE_RANK.get(max_price, 99)
        out = out[out["price_level"].map(lambda p: PRICE_RANK.get(p, 99) <= cap)]
    if min_rating is not None:
        out = out[out["google_rating"].fillna(-1) >= min_rating]
    if max_calories is not None:
        out = out[out["avg_calories_100g"].fillna(1e9) <= max_calories]
    if nutriscore is not None:
        out = out[out["common_nutriscore"].isin(_as_list(nutriscore))]
    if sentiment is not None:
        out = out[out["sentiment_label"] == sentiment]
    if has_reviews:
        out = out[out["review_text"].fillna("").str.strip().astype(bool)]
    if sort_by in out.columns:
        out = out.sort_values(by=sort_by, ascending=ascending, na_position="last")
    if limit is not None: out = out.head(limit)
    return out.reset_index(drop=True)

print("filter_venues() ready.")


In [ ]:
# Example searches
display_cols = ["name", "neighborhood", "cuisine_key", "google_rating",
                "price_level", "avg_calories_100g", "common_nutriscore",
                "sentiment_label"]

print("Highly rated cheap pizza near NYU:")
display(filter_venues(df_final, cuisine="pizza",
        neighborhood="NYU / Greenwich Village",
        max_price="$$", min_rating=4.0, limit=10)[display_cols])

print("\nHealthy options across the city (<= 250 kcal/100g, Nutri-Score a or b):")
display(filter_venues(df_final, max_calories=250, nutriscore=["a", "b"],
        sort_by="avg_calories_100g", ascending=True, limit=15)[display_cols])

print("\nVenues with positive sentiment, sorted by rating:")
display(filter_venues(df_final, sentiment="POSITIVE", has_reviews=True,
        sort_by="google_rating", limit=10)[display_cols])


---

## 7. Visualizations

The geographic distribution uses **folium** (interactive Leaflet map rendered inline). All other charts use **seaborn**.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["text.parse_math"] = False  # so "$$" labels render as text
print("seaborn ready -", sns.__version__)


In [ ]:
# ============================================================
# 1. Map: interactive folium map over Manhattan tiles
# ============================================================
# folium ships with Colab; install if running locally:
try:
    import folium
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "folium", "--quiet"])
    import folium

PALETTE = ["red", "blue", "green", "purple", "orange", "darkred",
           "cadetblue", "darkgreen", "pink", "black"]
nbhds = sorted(df_final["neighborhood"].dropna().unique())
color_for = {n: PALETTE[i % len(PALETTE)] for i, n in enumerate(nbhds)}

center = [df_final["lat"].mean(), df_final["lng"].mean()]
m = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")

for _, row in df_final.dropna(subset=["lat", "lng"]).iterrows():
    rating = row.get("google_rating")
    rating_str = f"{rating:.1f}" if pd.notna(rating) else "-"
    price = row.get("price_level") or "-"
    cuisine = row.get("cuisine_key") or "-"
    nbhd = row.get("neighborhood") or "-"
    name = row["name"]
    popup_html = (f"<b>{name}</b><br>{nbhd}<br>"
                  f"Cuisine: {cuisine}<br>"
                  f"Rating: {rating_str} &nbsp;|&nbsp; Price: {price}")
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=5,
        color=color_for.get(nbhd, "gray"),
        fill=True, fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=260),
    ).add_to(m)

# Simple HTML legend
rows = "".join(
    f"<span style=\"color:{color_for[n]}\">&#9679;</span> {n}<br>"
    for n in nbhds
)
legend_html = (
    "<div style=\"position: fixed; bottom: 30px; left: 30px; z-index:9999;"
    " background: white; padding: 10px 12px; border:1px solid #888;"
    " border-radius:6px; font-size:13px; line-height:1.4;\">"
    "<b>Neighborhoods</b><br>" + rows + "</div>"
)
m.get_root().html.add_child(folium.Element(legend_html))

m  # render inline


In [ ]:
# 2. Cuisine distribution (top 15) - countplot
top_cuisines = df_final["cuisine_key"].value_counts().head(15).index.tolist()

fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(
    data=df_final[df_final["cuisine_key"].isin(top_cuisines)],
    y="cuisine_key", hue="cuisine_key",
    order=top_cuisines, palette="viridis", legend=False, ax=ax)
ax.set_title("Most common cuisines across all neighborhoods")
ax.set_xlabel("Number of venues")
ax.set_ylabel("Cuisine")
plt.tight_layout(); plt.show()


In [ ]:
# 3. Google rating distribution per neighborhood - boxplot
df_rated = df_final[df_final["google_rating"].notna()].copy()

if len(df_rated) > 0:
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.boxplot(data=df_rated, x="neighborhood", y="google_rating",
                hue="neighborhood", palette="Set2", legend=False, ax=ax)
    sns.stripplot(data=df_rated, x="neighborhood", y="google_rating",
                  color="black", alpha=0.4, size=3, ax=ax)
    ax.set_title("Google rating distribution by neighborhood")
    ax.set_ylabel("Google rating (1.0 - 5.0)")
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout(); plt.show()
else:
    print("No rated venues to plot (Google API key likely missing).")


In [ ]:
# 4. Average nutrition per cuisine - grouped barplot
nut_cols = ["avg_calories_100g", "avg_fat_100g", "avg_protein_100g",
            "avg_carbs_100g", "avg_sugar_100g"]
cuisine_nut = (df_final.dropna(subset=["avg_calories_100g"])
               .drop_duplicates(subset=["cuisine_key"])
               [["cuisine_key"] + nut_cols]
               .sort_values("avg_calories_100g", ascending=False))
cuisine_nut_long = cuisine_nut.melt(id_vars="cuisine_key", value_vars=nut_cols,
                                    var_name="nutrient", value_name="grams_per_100g")
fig, ax = plt.subplots(figsize=(13, 7))
sns.barplot(data=cuisine_nut_long, x="cuisine_key", y="grams_per_100g",
            hue="nutrient", palette="rocket", ax=ax)
ax.set_title("Average nutrition per 100 g by cuisine")
ax.set_xlabel(""); ax.set_ylabel("Per 100 g")
plt.xticks(rotation=30, ha="right")
ax.legend(title="", loc="upper right")
plt.tight_layout(); plt.show()


In [ ]:
# 5. Sentiment breakdown - countplot per neighborhood
df_sent = df_final[df_final["sentiment_label"].notna()].copy()
if len(df_sent) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.countplot(data=df_sent, x="neighborhood", hue="sentiment_label",
                  palette={"POSITIVE": "#4caf50", "NEGATIVE": "#e53935"}, ax=ax)
    ax.set_title("Review sentiment by neighborhood")
    ax.set_xlabel(""); ax.set_ylabel("Number of venues")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout(); plt.show()
else:
    print("No sentiment data (HuggingFace token missing or no reviews).")


In [ ]:
# 6. Price level distribution - countplot
df_priced = df_final[df_final["price_level"].notna() &
                     (df_final["price_level"] != "")].copy()
if len(df_priced) > 0:
    order = ["$", "$$", "$$$", "$$$$"]
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.countplot(data=df_priced, x="price_level", order=order,
                  hue="price_level", palette="mako", legend=False, ax=ax)
    ax.set_title("Price level distribution (Google data)")
    ax.set_xlabel("Price tier"); ax.set_ylabel("Number of venues")
    plt.tight_layout(); plt.show()
else:
    print("No price data to plot.")


In [ ]:
# 7. Calories vs rating - scatter, sized by review count
df_xy = df_final.dropna(subset=["google_rating", "avg_calories_100g"]).copy()
if len(df_xy) > 0:
    top_c = df_xy["cuisine_key"].value_counts().head(8).index.tolist()
    df_xy = df_xy[df_xy["cuisine_key"].isin(top_c)]
    fig, ax = plt.subplots(figsize=(11, 7))
    sns.scatterplot(data=df_xy, x="avg_calories_100g", y="google_rating",
                    hue="cuisine_key", size="rating_count", sizes=(20, 300),
                    alpha=0.7, palette="Set1", ax=ax)
    ax.set_title("Calories per 100 g vs Google rating")
    ax.set_xlabel("Avg calories per 100g (cuisine-level)")
    ax.set_ylabel("Google rating")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout(); plt.show()
else:
    print("Not enough overlapping data for scatter.")


---

## What's next (Milestone 5)

- Build a Flask or Streamlit UI on top of `filter_venues()` and the folium map.
- Add per-venue nutrition (Edamam / USDA) instead of cuisine-level averages.
- Cluster restaurants by feature similarity (price + cuisine + nutrition).
